# Build Legal Knowledge Graph
Runs gemma-2-9b-it (4-bit NF4) to extract entity triples from Swiss law articles.
Saves `entities.json` + `triples.jsonl` to `/kaggle/working/kg/`.

In [ ]:
import os
CORPUS       = '/kaggle/input/llm-agentic-legal-information-retrieval/laws_de.csv'
KG_DIR       = '/kaggle/working/kg'
SAMPLE_SEQ   = 500   # first N rows (early SR numbers)
SAMPLE_RAND  = 500   # random N rows from the rest
os.makedirs(KG_DIR, exist_ok=True)

In [ ]:
from kaggle_secrets import UserSecretsClient
import subprocess
token = UserSecretsClient().get_secret('HF_TOKEN')
subprocess.run(['huggingface-cli', 'login', '--token', token], check=True)

In [ ]:
!pip install -q networkx

## Data models

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field

@dataclass
class Entity:
    name: str
    type: str
    abbreviations: list[str] = field(default_factory=list)
    aliases: list[str] = field(default_factory=list)

@dataclass
class Triple:
    subject: str
    relation: str
    relation_explain: str      # why this relation was chosen
    object: str
    source_citation: str
    confidence: float = 1.0

@dataclass
class ExtractionResult:
    citation: str
    entities: list[Entity]
    triples: list[Triple]
    abbreviations: list[tuple[str, str]]
    new_relations: list[dict]  # [{"name": ..., "definition": ...}]

## Store

In [ ]:
import json
from pathlib import Path
import networkx as nx

class KGStore:
    def __init__(self, db_dir):
        self.db_dir = Path(db_dir)
        self.db_dir.mkdir(parents=True, exist_ok=True)
        self._entities_path = self.db_dir / 'entities.json'
        self._triples_path  = self.db_dir / 'triples.jsonl'
        self._entities: dict[str, dict] = {}
        self._alias_index: dict[str, str] = {}
        self.graph = nx.MultiDiGraph()
        self._load()

    def _load(self):
        if self._entities_path.exists():
            self._entities = json.loads(self._entities_path.read_text(encoding='utf-8'))
            for name, info in self._entities.items():
                self.graph.add_node(name, type=info['type'])
                for a in info.get('aliases', []):
                    self._alias_index[a] = name
                for a in info.get('abbreviations', []):
                    self._alias_index[a] = name
        if self._triples_path.exists():
            for line in self._triples_path.read_text(encoding='utf-8').splitlines():
                if not line.strip(): continue
                t = json.loads(line)
                self.graph.add_edge(t['subject'], t['object'],
                    relation=t['relation'],
                    relation_explain=t.get('relation_explain', ''),
                    source=t.get('source_citation', ''),
                    confidence=t.get('confidence', 1.0))

    def save(self):
        self._entities_path.write_text(
            json.dumps(self._entities, ensure_ascii=False, indent=2), encoding='utf-8')
        lines = []
        for s, o, d in self.graph.edges(data=True):
            lines.append(json.dumps({
                'subject': s,
                'relation': d.get('relation', ''),
                'relation_explain': d.get('relation_explain', ''),
                'object': o,
                'source_citation': d.get('source', ''),
                'confidence': d.get('confidence', 1.0),
            }, ensure_ascii=False))
        self._triples_path.write_text('\n'.join(lines), encoding='utf-8')

    def upsert_entity(self, entity: Entity):
        if entity.name not in self._entities:
            self._entities[entity.name] = {'type': entity.type, 'abbreviations': [], 'aliases': []}
            self.graph.add_node(entity.name, type=entity.type)
        info = self._entities[entity.name]
        for a in entity.aliases:
            if a not in info['aliases']: info['aliases'].append(a)
            self._alias_index[a] = entity.name
        for a in entity.abbreviations:
            if a not in info['abbreviations']: info['abbreviations'].append(a)
            self._alias_index[a] = entity.name

    def add_triple(self, triple: Triple):
        for name in (triple.subject, triple.object):
            if name not in self._entities:
                self._entities[name] = {'type': 'concept', 'abbreviations': [], 'aliases': []}
                self.graph.add_node(name, type='concept')
        self.graph.add_edge(triple.subject, triple.object,
            relation=triple.relation,
            relation_explain=triple.relation_explain,
            source=triple.source_citation,
            confidence=triple.confidence)

    def add_result(self, result: ExtractionResult):
        for e in result.entities: self.upsert_entity(e)
        for short, full in result.abbreviations: self._ensure_abbrev_link(short, full)
        for t in result.triples: self.add_triple(t)

    def resolve(self, name):
        if name in self._entities: return name
        return self._alias_index.get(name)

    def _ensure_abbrev_link(self, short, full):
        full_c = self.resolve(full)
        short_c = self.resolve(short)
        if full_c:
            info = self._entities[full_c]
            if short not in info['abbreviations']: info['abbreviations'].append(short)
            self._alias_index[short] = full_c
        elif short_c:
            info = self._entities[short_c]
            if full not in info['aliases']: info['aliases'].append(full)
            self._alias_index[full] = short_c
        else:
            self._entities[full] = {'type': 'law', 'abbreviations': [short], 'aliases': []}
            self.graph.add_node(full, type='law')
            self._alias_index[short] = full

    def merge_equivalent_entities(self):
        merges = 0
        for alias, primary in list(self._alias_index.items()):
            if alias == primary or alias not in self._entities: continue
            dup_info = self._entities.pop(alias)
            pinfo = self._entities[primary]
            for a in dup_info.get('aliases', []):
                if a not in pinfo['aliases'] and a != primary: pinfo['aliases'].append(a)
                self._alias_index[a] = primary
            for a in dup_info.get('abbreviations', []):
                if a not in pinfo['abbreviations'] and a != primary: pinfo['abbreviations'].append(a)
                self._alias_index[a] = primary
            if self.graph.has_node(alias):
                for _, t, d in list(self.graph.out_edges(alias, data=True)):
                    self.graph.add_edge(primary, t, **d)
                for s, _, d in list(self.graph.in_edges(alias, data=True)):
                    self.graph.add_edge(s, primary, **d)
                self.graph.remove_node(alias)
            merges += 1
        return merges

    def stats(self):
        return {'entities': len(self._entities), 'aliases': len(self._alias_index),
                'graph_nodes': self.graph.number_of_nodes(),
                'graph_edges': self.graph.number_of_edges()}

## Linker — seed known Swiss-law abbreviations

In [ ]:
import re

KNOWN_ABBREVS = {
    'BV': 'Bundesverfassung', 'OR': 'Obligationenrecht', 'ZGB': 'Zivilgesetzbuch',
    'StGB': 'Strafgesetzbuch', 'DBG': 'Bundesgesetz über die direkte Bundessteuer',
    'MWSTG': 'Mehrwertsteuergesetz',
    'AHVG': 'Bundesgesetz über die Alters- und Hinterlassenenversicherung',
    'IVG': 'Bundesgesetz über die Invalidenversicherung',
    'KVG': 'Bundesgesetz über die Krankenversicherung',
    'UVG': 'Bundesgesetz über die Unfallversicherung',
    'BVG': 'Bundesgesetz über die berufliche Vorsorge',
    'FINMA': 'Eidgenössische Finanzmarktaufsicht',
    'BAZL': 'Bundesamt für Zivilluftfahrt', 'ASTRA': 'Bundesamt für Strassen',
    'BAFU': 'Bundesamt für Umwelt', 'BFE': 'Bundesamt für Energie',
    'BAG': 'Bundesamt für Gesundheit', 'SECO': 'Staatssekretariat für Wirtschaft',
    'EJPD': 'Eidgenössisches Justiz- und Polizeidepartement',
    'EFD': 'Eidgenössisches Finanzdepartement',
    'UVEK': 'Eidgenössisches Departement für Umwelt, Verkehr, Energie und Kommunikation',
}

def seed_known_abbrevs(store):
    for short, full in KNOWN_ABBREVS.items():
        store._ensure_abbrev_link(short, full)

def link_result(store, result: ExtractionResult):
    for short, full in result.abbreviations:
        store._ensure_abbrev_link(short, full)
    for triple in result.triples:
        c = store.resolve(triple.subject)
        if c: triple.subject = c
        c = store.resolve(triple.object)
        if c: triple.object = c

In [ ]:
import json, re, torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME     = 'google/gemma-2-9b-it'
RELATIONS_PATH = Path(KG_DIR) / 'relations.json'

# ── Known relations — grows as Gemma discovers new ones ───────────────────────
RELATIONS: dict[str, str] = {
    "requires":   "subject IMPOSES an obligation ON object. e.g. 'Arbeitgeber requires Beitragspflicht'",
    "regulates":  "subject GOVERNS or SETS RULES FOR object. e.g. 'KVG regulates Krankenversicherung'",
    "issued_by":  "subject (a rule/ordinance) was ENACTED BY object, which must be an AUTHORITY (Bundesrat, Parlament, FINMA). "
                  "✗ WRONG: article→law, law→canton. ✓ RIGHT: Ausführungsbestimmungen→Bundesrat",
    "cites":      "subject EXPLICITLY REFERENCES a DIFFERENT LAW by name or abbreviation (OR, ZGB, StGB, KVG...). "
                  "✗ WRONG: pointing at an entity, person, or same-law article. "
                  "✓ RIGHT: 'Schadensersatzanspruch cites Obligationenrecht' because the text says 'gemäss OR'",
    "exempts":    "subject grants an exception or exclusion from object",
    "modifies":   "subject amends, replaces, or changes object",
    "defines":    "subject gives the legal definition of object. e.g. 'gilt als', 'im Sinne dieses Gesetzes'",
    "applies_to": "subject's scope, rule, or obligation covers object",
}

if RELATIONS_PATH.exists():
    RELATIONS.update(json.loads(RELATIONS_PATH.read_text(encoding='utf-8')))
    print(f'Loaded {len(RELATIONS)} relations from {RELATIONS_PATH}')
else:
    print(f'Starting with {len(RELATIONS)} seed relations')


def _relations_block() -> str:
    return '\n'.join(f'- {name}: {defn}' for name, defn in RELATIONS.items())


PROMPT_TEMPLATE = """You are a legal knowledge graph builder for Swiss federal law (German).
Extract named entities and relationships from the article below.

Citation: {citation}
Title: {title}
Text: {text}

Entity types:
- law: a statute, ordinance, or treaty (e.g. "Obligationenrecht", "OR", "ZGB", "KVG")
- authority: an official body or government office (e.g. "Bundesrat", "FINMA", "Bundesgericht")
- concept: a specific legal term (e.g. "Versicherungspflicht", "Schadensersatzanspruch", "Verjährungsfrist")
- person: a legal role (e.g. "Arbeitnehmer", "Antragsteller", "Versicherter")
- procedure: a formal legal process (e.g. "Einsprache", "Bewilligung", "Volksinitiative")

Known relations:
{relations_block}

If NONE of the above relations fit, you may invent a new one. Add it to "new_relations" with its definition.

Each triple MUST include a "relation_explain" field: one sentence explaining WHY you chose this relation for this specific text.

---
EXAMPLE 1 — issued_by (authority enacted the rule) + cites (cross-law reference)
Input: "Der Bundesrat erlässt die Ausführungsbestimmungen. Er kann dabei auf das Obligationenrecht (OR) verweisen."
Output:
{{
  "entities": [
    {{"name": "Bundesrat", "type": "authority", "abbreviations": [], "aliases": []}},
    {{"name": "Ausführungsbestimmungen", "type": "concept", "abbreviations": [], "aliases": []}},
    {{"name": "Obligationenrecht", "type": "law", "abbreviations": ["OR"], "aliases": []}}
  ],
  "triples": [
    {{
      "subject": "Ausführungsbestimmungen", "relation": "issued_by",
      "relation_explain": "The text says 'Der Bundesrat erlässt' — Bundesrat is the enacting authority.",
      "object": "Bundesrat", "confidence": 0.95
    }},
    {{
      "subject": "Ausführungsbestimmungen", "relation": "cites",
      "relation_explain": "The text explicitly references 'das Obligationenrecht (OR)', a different law.",
      "object": "Obligationenrecht", "confidence": 0.90
    }}
  ],
  "new_relations": [],
  "abbreviations": [{{"short": "OR", "full": "Obligationenrecht"}}]
}}

---
EXAMPLE 2 — cites bridges two laws; requires + applies_to
Input (Art. 955 ZGB): "Der Kanton haftet für den Schaden aus Fehlern des Grundbuchamtes. Der Schadensersatzanspruch verjährt gemäss den Vorschriften des Obligationenrechts (OR)."
Output:
{{
  "entities": [
    {{"name": "Kanton", "type": "authority", "abbreviations": [], "aliases": []}},
    {{"name": "Grundbuchamt", "type": "authority", "abbreviations": [], "aliases": []}},
    {{"name": "Haftung des Kantons", "type": "concept", "abbreviations": [], "aliases": []}},
    {{"name": "Schadensersatzanspruch", "type": "concept", "abbreviations": [], "aliases": []}},
    {{"name": "Obligationenrecht", "type": "law", "abbreviations": ["OR"], "aliases": []}}
  ],
  "triples": [
    {{
      "subject": "Kanton", "relation": "requires",
      "relation_explain": "The text says 'haftet' — the canton is obligated to bear liability, imposing a legal duty.",
      "object": "Haftung des Kantons", "confidence": 0.95
    }},
    {{
      "subject": "Schadensersatzanspruch", "relation": "cites",
      "relation_explain": "The text says 'verjährt gemäss den Vorschriften des Obligationenrechts (OR)' — explicitly naming OR as the governing law for the claim's limitation period.",
      "object": "Obligationenrecht", "confidence": 0.95
    }}
  ],
  "new_relations": [],
  "abbreviations": [{{"short": "OR", "full": "Obligationenrecht"}}]
}}

---
EXAMPLE 3 — applies_to + aliases; new relation invented
Input (Art. 60 OR): "Der Anspruch auf Schadensersatz oder Genugtuung verjährt mit Ablauf von einem Jahr. Bei unerlaubten Handlungen beträgt die absolute Verjährungsfrist zehn Jahre."
Output:
{{
  "entities": [
    {{"name": "Schadensersatzanspruch", "type": "concept", "abbreviations": [], "aliases": ["Anspruch auf Schadensersatz"]}},
    {{"name": "Verjährungsfrist", "type": "concept", "abbreviations": [], "aliases": []}},
    {{"name": "unerlaubte Handlung", "type": "concept", "abbreviations": [], "aliases": []}}
  ],
  "triples": [
    {{
      "subject": "Verjährungsfrist", "relation": "applies_to",
      "relation_explain": "The text sets a 1-year limitation period specifically for Schadensersatzanspruch, so Verjährungsfrist's scope covers this claim type.",
      "object": "Schadensersatzanspruch", "confidence": 0.95
    }},
    {{
      "subject": "unerlaubte Handlung", "relation": "triggers",
      "relation_explain": "The text states a 10-year absolute period 'bei unerlaubten Handlungen' — the wrongful act triggers the extended limitation.",
      "object": "Verjährungsfrist", "confidence": 0.85
    }}
  ],
  "new_relations": [
    {{"name": "triggers", "definition": "subject (an event or act) causes or activates object (a legal consequence or period)"}}
  ],
  "abbreviations": []
}}

---
Now extract from the article above. Return ONLY valid JSON — no explanation, no markdown.
Rules:
- Only extract SPECIFIC legal terms — not generic words like "Schaden", "Person", "Recht"
- Use compound terms: "Schadensersatzanspruch" not "Schaden", "Verjährungsfrist" not "Frist"
- Canonical German nominative singular form
- issued_by: ONLY when an authority enacted something — never article→law
- cites: ONLY when text explicitly names a DIFFERENT law by abbreviation or full name
- relation_explain: required for every triple — quote the specific text that justifies the relation
- aliases: fill when the same concept appears under two phrasings in this text
- new_relations: add if and only if no known relation fits; include name + definition
- Only include triples with confidence >= 0.7
- Empty lists if nothing to extract"""


def _build_prompt(citation, title, text, feedback: str = ''):
    base = PROMPT_TEMPLATE.format(
        citation=citation, title=title[:300], text=text[:1500],
        relations_block=_relations_block(),
    )
    if feedback:
        base += f'\n\nIMPORTANT — A reviewer rejected your previous attempt with this feedback:\n{feedback}\nAddress these issues in your new output.'
    return base


def _parse(raw, citation, regex_abbrevs):
    raw = re.sub(r'^```(?:json)?\s*', '', raw.strip())
    raw = re.sub(r'\s*```$', '', raw.strip())
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if not match:
        return ExtractionResult(citation=citation, entities=[], triples=[],
                                abbreviations=regex_abbrevs, new_relations=[])
    try:
        data = json.loads(match.group())
    except json.JSONDecodeError:
        return ExtractionResult(citation=citation, entities=[], triples=[],
                                abbreviations=regex_abbrevs, new_relations=[])

    entities = [
        Entity(name=(e.get('name') or '').strip(), type=e.get('type') or 'concept',
               abbreviations=e.get('abbreviations') or [], aliases=e.get('aliases') or [])
        for e in data.get('entities', []) if (e.get('name') or '').strip()
    ]
    triples = [
        Triple(
            subject=(t.get('subject') or '').strip(),
            relation=t.get('relation') or 'relates_to',
            relation_explain=(t.get('relation_explain') or '').strip(),
            object=(t.get('object') or '').strip(),
            source_citation=citation,
            confidence=float(t.get('confidence') or 1.0),
        )
        for t in data.get('triples', [])
        if (t.get('subject') or '').strip() and (t.get('object') or '').strip()
    ]
    llm_abbrevs = [
        (a['short'].strip(), a['full'].strip())
        for a in data.get('abbreviations', [])
        if (a.get('short') or '').strip() and (a.get('full') or '').strip()
    ]
    merged = {a[0]: a for a in regex_abbrevs + llm_abbrevs}
    new_relations = [
        {'name': r['name'].strip(), 'definition': r['definition'].strip()}
        for r in data.get('new_relations', [])
        if (r.get('name') or '').strip() and (r.get('definition') or '').strip()
    ]
    return ExtractionResult(
        citation=citation, entities=entities, triples=triples,
        abbreviations=list(merged.values()), new_relations=new_relations,
    )


def _run_model(tokenizer, model, prompt_text, max_new_tokens):
    messages = [{'role': 'user', 'content': prompt_text}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             temperature=None, top_p=None,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()


class Extractor:
    def __init__(self):
        print(f'Loading {MODEL_NAME}...')
        quant_cfg = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4')
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        self.model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, quantization_config=quant_cfg, device_map='auto')
        self.model.eval()
        print('Model ready')

    def extract(self, citation, title, text, feedback: str = ''):
        regex_abbrevs = _extract_abbrevs_regex(title + ' ' + text)
        raw = _run_model(self.tokenizer, self.model,
                         _build_prompt(citation, title, text, feedback),
                         max_new_tokens=600)
        return _parse(raw, citation, regex_abbrevs), raw


def _extract_abbrevs_regex(text):
    pattern = re.compile(r'([A-ZÄÖÜ][^\(\)]{5,80}?)\s+\(([A-ZÄÖÜa-z][A-Za-z0-9\-]{1,15})\)')
    found = {}
    for full, short in pattern.findall(text):
        full, short = full.strip(), short.strip()
        if short.isupper() or re.match(r'^[A-Z][a-z]?[A-Z]', short):
            found[short] = (short, full)
    return list(found.values())


print('Extractor defined. Relations:', list(RELATIONS.keys()))


## Judge — LLM-as-critic for extracted triples

In [ ]:
JUDGE_PROMPT_TEMPLATE = """You are auditing legal knowledge graph triples extracted from a Swiss law article.

Citation: {citation}
Text: {text}

Extracted triples:
{triples_json}

For each triple evaluate:
1. Subject and object are SPECIFIC legal terms actually present in the text (not generic words like "Recht", "Person", "Schaden").
2. Relation type is correct:
   - issued_by → ONLY if an AUTHORITY (Bundesrat, FINMA, Parlament) enacted the subject. WRONG: article → parent law.
   - cites → ONLY if a DIFFERENT LAW is explicitly named by abbreviation (OR, ZGB, StGB...). WRONG: internal article refs, entity names.
   - regulates → subject governs/sets rules for object. WRONG: using this as a catch-all fallback.
3. relation_explain quotes actual text from the article.

Return ONLY valid JSON, no markdown:
{{
  "verdict": "accept" or "reject",
  "issues": [
    {{"index": 0, "problem": "short description", "fix": "what to change"}}
  ],
  "feedback": "One short paragraph of specific actionable feedback for the extractor, referencing the actual text."
}}

Reject ONLY for serious errors (wrong relation type, hallucinated entities, internal refs mislabelled as cites).
Accept if triples are reasonable even if imperfect. Empty issues list on accept."""


class Judge:
    """Reuses the same loaded Gemma model to critique extractor output."""

    def __init__(self, extractor: 'Extractor'):
        self.tokenizer = extractor.tokenizer
        self.model     = extractor.model

    def review(self, citation: str, text: str, result: ExtractionResult):
        """
        Returns (verdict, issues, feedback, raw_output).
        verdict: 'accept' | 'reject'
        issues:  list[dict]  — per-triple problems
        feedback: str        — sent back to extractor on retry
        """
        if not result.triples:
            return 'accept', [], '', ''

        triples_json = json.dumps(
            [{'index': i,
              'subject': t.subject, 'relation': t.relation,
              'relation_explain': t.relation_explain, 'object': t.object}
             for i, t in enumerate(result.triples)],
            ensure_ascii=False, indent=2
        )
        prompt_text = JUDGE_PROMPT_TEMPLATE.format(
            citation=citation,
            text=text[:1000],
            triples_json=triples_json,
        )
        messages = [{'role': 'user', 'content': prompt_text}]
        prompt = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(
                **inputs, max_new_tokens=300, do_sample=False,
                temperature=None, top_p=None,
                pad_token_id=self.tokenizer.eos_token_id)
        raw = self.tokenizer.decode(
            out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

        verdict, issues, feedback = self._parse_review(raw)
        return verdict, issues, feedback, raw

    def _parse_review(self, raw: str):
        raw = re.sub(r'^```(?:json)?\s*', '', raw.strip())
        raw = re.sub(r'\s*```$', '', raw.strip())
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        if not match:
            return 'accept', [], ''
        try:
            data = json.loads(match.group())
        except json.JSONDecodeError:
            return 'accept', [], ''
        verdict   = 'reject' if str(data.get('verdict', '')).lower() == 'reject' else 'accept'
        issues    = data.get('issues', []) or []
        feedback  = (data.get('feedback') or '').strip()
        return verdict, issues, feedback


print('Judge defined')


## Load corpus

In [ ]:
import pandas as pd

df = pd.read_csv(CORPUS, low_memory=False)
df[['citation','title','text']] = df[['citation','title','text']].fillna('')
print(f'Total rows: {len(df):,}')

# first 500 sequential (covers early SR numbers) + 500 random from the rest
first_500 = df.iloc[:500]
rest      = df.iloc[500:]
random_500 = rest.sample(500, random_state=42)

df_sample = pd.concat([first_500, random_500]).reset_index(drop=True)
print(f'Sample: {len(first_500)} sequential + {len(random_500)} random = {len(df_sample)} total')
print(f'Laws covered: {df_sample["citation"].str.extract(r"(\S+)$")[0].nunique()} unique')

rows = df_sample[['citation', 'title', 'text']].to_dict('records')
print(f'Processing {len(rows):,} articles')

## Init store, linker, extractor

In [ ]:
store = KGStore(KG_DIR)
seed_known_abbrevs(store)
extractor = Extractor()
judge = Judge(extractor)
print('Ready')

## Extract triples

In [ ]:
SAVE_EVERY    = 50
TRACES_PATH   = Path(KG_DIR) / 'traces.jsonl'
REVIEW_PATH   = Path(KG_DIR) / 'traces_review.jsonl'   # doubles-rejected, for later learning

# ── Resume: skip already-processed citations ──────────────────────────────────
done_citations: set[str] = set()
if TRACES_PATH.exists():
    for line in TRACES_PATH.read_text(encoding='utf-8').splitlines():
        if line.strip():
            done_citations.add(json.loads(line)['citation'])
if REVIEW_PATH.exists():
    for line in REVIEW_PATH.read_text(encoding='utf-8').splitlines():
        if line.strip():
            done_citations.add(json.loads(line)['citation'])

rows_to_process = [r for r in rows if r.get('citation', '') not in done_citations]
print(f'Already done: {len(done_citations)}, remaining: {len(rows_to_process)}')

def _triple_dicts(result):
    return [{'subject': t.subject, 'relation': t.relation,
             'relation_explain': t.relation_explain,
             'object': t.object, 'confidence': t.confidence}
            for t in result.triples]

traces_file = TRACES_PATH.open('a', encoding='utf-8')
review_file = REVIEW_PATH.open('a', encoding='utf-8')

stats_counter = {'accept_r1': 0, 'accept_r2': 0, 'rejected': 0}

for i, row in enumerate(rows_to_process):
    citation = row.get('citation', '')
    title    = str(row.get('title', '') or '')
    text     = str(row.get('text', '') or '')

    # ── Round 1: extract ──────────────────────────────────────────────────────
    result1, raw1 = extractor.extract(citation=citation, title=title, text=text)
    verdict1, issues1, feedback1, raw_judge1 = judge.review(citation, text, result1)

    if verdict1 == 'accept':
        final_result = result1
        judge_outcome = 'accept_r1'
        round2 = None
    else:
        # ── Round 2: re-extract with judge feedback ───────────────────────────
        result2, raw2 = extractor.extract(citation=citation, title=title, text=text,
                                          feedback=feedback1)
        verdict2, issues2, feedback2, raw_judge2 = judge.review(citation, text, result2)
        round2 = {
            'raw_output':    raw2,
            'entities':      [{'name': e.name, 'type': e.type} for e in result2.entities],
            'triples':       _triple_dicts(result2),
            'new_relations': result2.new_relations,
            'judge_verdict': verdict2,
            'judge_issues':  issues2,
            'judge_raw':     raw_judge2,
        }
        if verdict2 == 'accept':
            final_result = result2
            judge_outcome = 'accept_r2'
        else:
            final_result = None
            judge_outcome = 'rejected'

    stats_counter[judge_outcome] += 1

    # ── Register new relations from accepted result ───────────────────────────
    if final_result is not None:
        for nr in final_result.new_relations:
            if nr['name'] not in RELATIONS:
                RELATIONS[nr['name']] = nr['definition']
                print(f'  [new relation] {nr["name"]}: {nr["definition"]}')
        link_result(store, final_result)
        store.add_result(final_result)

    # ── Build trace ───────────────────────────────────────────────────────────
    trace = {
        'citation':      citation,
        'title':         title[:200],
        'text_input':    text[:500],
        'judge_outcome': judge_outcome,
        'round1': {
            'raw_output':    raw1,
            'entities':      [{'name': e.name, 'type': e.type} for e in result1.entities],
            'triples':       _triple_dicts(result1),
            'new_relations': result1.new_relations,
            'judge_verdict': verdict1,
            'judge_issues':  issues1,
            'judge_raw':     raw_judge1,
            'judge_feedback': feedback1,
        },
        'round2': round2,
    }

    if judge_outcome == 'rejected':
        review_file.write(json.dumps(trace, ensure_ascii=False) + '\n')
    else:
        traces_file.write(json.dumps(trace, ensure_ascii=False) + '\n')

    if (i + 1) % 10 == 0:
        traces_file.flush()
        review_file.flush()
        r1, r2, rej = stats_counter['accept_r1'], stats_counter['accept_r2'], stats_counter['rejected']
        print(f'  {i+1}/{len(rows_to_process)}  '
              f'accept_r1={r1} accept_r2={r2} rejected={rej}  '
              f'entities={len(store._entities)}  edges={store.graph.number_of_edges()}')

    if (i + 1) % SAVE_EVERY == 0:
        store.save()
        RELATIONS_PATH.write_text(json.dumps(RELATIONS, ensure_ascii=False, indent=2), encoding='utf-8')
        print(f'  [checkpoint saved]')

traces_file.close()
review_file.close()

r1, r2, rej = stats_counter['accept_r1'], stats_counter['accept_r2'], stats_counter['rejected']
print(f'\nDone: accept_r1={r1} ({100*r1/len(rows_to_process):.0f}%)  '
      f'accept_r2={r2} ({100*r2/len(rows_to_process):.0f}%)  '
      f'rejected={rej} ({100*rej/len(rows_to_process):.0f}%)')
print(f'Final relations ({len(RELATIONS)}):')
for name, defn in RELATIONS.items():
    print(f'  {name}: {defn[:80]}')


## Post-process and save

In [ ]:
n_merges = store.merge_equivalent_entities()
print(f'Merged {n_merges} duplicate entities')

store.save()

stats = store.stats()
print('\nKnowledge graph stats:')
for k, v in stats.items():
    print(f'  {k}: {v:,}')

print(f'\nSaved to {KG_DIR}/')

## Quick sanity check — sample triples

In [ ]:
import random
edges = list(store.graph.edges(data=True))
for s, o, d in random.sample(edges, min(10, len(edges))):
    print(f'{s} --[{d["relation"]}]--> {o}  (src: {d["source"][:40]})')